# Calorvia · Carnet de travail Text-to-SQL

**Z4 · Point de reprise.** Les quatre tests de permissions des sources ont réussi. Le diagnostic du journal a renvoyé le code 3 : dans `calorvia_rag`, le compte `calorvia_app` n’a pas le droit de créer le schéma absent. L’essai s’est arrêté avant MCP et Qwen.

## 1. Préparer le journal du labo 2 (CPU)

La cellule utilise le serveur PostgreSQL déjà restauré sur 127.0.0.1:5433. Elle crée une base et un compte réservés au journal. Les tables métier de `compta` et le serveur du RAG sur 5432 sont hors du périmètre des écritures. Le rôle administrateur du cours sert uniquement à cette préparation. Le DSN du journal reste dans un fichier privé local, réutilisable après redémarrage du noyau.

In [ ]:
import os
import secrets
import sys
from pathlib import Path
from tempfile import NamedTemporaryFile

import psycopg
from dotenv import dotenv_values
from psycopg import sql

cwd = Path.cwd().resolve()
racine = next(p for p in (cwd, *cwd.parents) if (p / "text2sql/journal.sql").is_file())
sys.path.insert(0, str(racine))
from text2sql.journal import Journal

def preparer_journal():
    """Crée ou réutilise la base du journal sur le serveur comptable du labo 2."""
    config = Path.home() / "labo/calorvia_t2sql/sources_calorvia/docker/.env"
    cours = dotenv_values(config, interpolate=False)
    role, base = "text2sql_writer", "calorvia_text2sql_journal"
    serveur = dict(host="127.0.0.1", port=5433, connect_timeout=5)
    dossier = racine / "data/text2sql"
    dossier.mkdir(parents=True, exist_ok=True)
    fichier = dossier / "journal_z4.env"
    prefixe = f"postgresql://{role}:"
    suffixe = f"@127.0.0.1:5433/{base}"

    with psycopg.connect(**serveur, dbname="postgres", user=cours["PG_ADMIN_USER"],
                         password=cours["PG_ADMIN_PASSWORD"], autocommit=True) as admin:
        admin.execute("SELECT pg_advisory_lock(116207, 5433)")
        proprietaire = admin.execute(
            "SELECT pg_get_userbyid(datdba) FROM pg_database WHERE datname=%s", (base,)
        ).fetchone()
        flags = admin.execute(
            "SELECT rolcanlogin, rolsuper, rolcreatedb, rolcreaterole, rolreplication,"
            " rolbypassrls FROM pg_roles WHERE rolname=%s", (role,)
        ).fetchone()
        if proprietaire not in (None, (role,)) or flags not in (None, (True, False, False, False, False, False)):
            raise RuntimeError("Un objet de même nom existe avec un propriétaire ou des droits inattendus.")
        if not fichier.exists():
            if flags or proprietaire:
                raise RuntimeError("Configuration privée manquante pour un journal existant. Récupérer son fichier journal_z4.env.")
            # Publication atomique, permissions 0600 ; le secret survit à une interruption.
            with NamedTemporaryFile(mode="w", dir=dossier, delete=False) as tmp:
                tmp.write(f"T2SQL_JOURNAL_DSN={prefixe}{secrets.token_hex(24)}{suffixe}\n")
            try:
                os.link(tmp.name, fichier)
            finally:
                Path(tmp.name).unlink()
        dsn = dotenv_values(fichier, interpolate=False).get("T2SQL_JOURNAL_DSN") or ""
        if not dsn.startswith(prefixe) or not dsn.endswith(suffixe):
            raise ValueError("Destination inattendue dans le fichier privé du journal.")
        secret = dsn[len(prefixe):-len(suffixe)]
        if len(secret) != 48 or any(c not in "0123456789abcdef" for c in secret):
            raise ValueError("Format du secret du journal inattendu.")
        if flags is None:
            verifier = admin.pgconn.encrypt_password(secret.encode(), role.encode(), b"scram-sha-256").decode()
            admin.execute(sql.SQL("CREATE ROLE {} LOGIN PASSWORD {}").format(sql.Identifier(role), sql.Literal(verifier)))
        if proprietaire is None:
            admin.execute(sql.SQL("CREATE DATABASE {} OWNER {}").format(sql.Identifier(base), sql.Identifier(role)))
    with psycopg.connect(dsn, connect_timeout=5) as ecrivain:
        ecrivain.execute(sql.SQL("REVOKE ALL ON DATABASE {} FROM PUBLIC").format(sql.Identifier(base)))
    Journal(dsn).initialize()
    with psycopg.connect(dsn, connect_timeout=5) as ecrivain:
        identite = ecrivain.execute("SELECT current_database(), current_user").fetchone()
        ecrivain.execute("INSERT INTO text2sql_journal.events(question,source,sql,stage) "
                         "VALUES ('diagnostic','journal','','test')")
        ecrivain.rollback()
    os.environ.update(T2SQL_JOURNAL_DSN=dsn, T2SQL_SOURCE_PASSWORD=cours["AGENT_RO_PASSWORD"])
    print("Journal prêt |", *identite, "| écriture de contrôle annulée")

journal_pret = False
preparer_journal()
journal_pret = True

## 2. Une question via MCP et Qwen

Exécuter uniquement après « Journal prêt », dans le même noyau et entre deux traitements RAG. Cette cellule envoie une seule question à l’agent, sans ontologie, puis conserve sa réponse et ses appels d’outils. Le contrôle natif précédent sur `compta.ecriture` donne **20 562**. Comparer les lignes SQL et la réponse rédigée à ce total.

In [ ]:
import json
from datetime import datetime
from text2sql.agent import answer
from text2sql.config import Settings

if not globals().get("journal_pret"):
    raise RuntimeError("Exécuter d’abord la cellule du journal.")
os.environ.update(
    T2SQL_COMPTA_PORT="5433", T2SQL_ERP_PORT="14330", T2SQL_PAIE_PORT="5434",
    T2SQL_LLM_URL="http://127.0.0.1:8001/v1",
    T2SQL_LLM_MODEL="/home/bm/models/qwen3.5-9b/llama/Qwen3.5-9B-UD-Q4_K_XL.gguf",
)
question = "Combien d’écritures comptables la base de comptabilité contient-elle au total ?"
resultat = await answer(question, use_ontology=False, settings=Settings.load())
trace = racine / "data/text2sql" / f"premiere_question_z4_{datetime.now():%Y%m%d_%H%M%S_%f}.json"
trace.write_text(json.dumps({"question": question, **resultat}, default=str, indent=2,
                            ensure_ascii=False), encoding="utf-8")
print(resultat["answer"])
for execution in resultat["executions"]:
    print(execution["source"], execution["sql"], execution["rows"])
print("Outils :", [appel["tool"] for appel in resultat["tool_trace"]])
print("Trace :", trace)